<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 26 — פרספטרון: סכום משוקלל ולמידה מטעות
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
במקום לחפש שכנים, ננסה גבול החלטה המבוסס על משקולות שמשתנות בעקבות טעויות בדוגמאות אימון.

## מטרות
- לחשב החלטה.
- לעקוב אחרי עדכון.
- להבחין בין אימון להערכה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## המושגים הנחוצים
פרספטרון מחשב z=w·x+b וממיר אותו לתווית לפי סף. כאן נבחר הכלל שמופיע בשקף 3: תווית 1 כאשר z>=0, אחרת 0. עדכון משקולות ו־bias תלוי בטעות ובקצב הלמידה. זהו אימון של מסווג ליניארי, שאינו פותר XOR במרחב המקורי.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## דוגמה פתורה לפני קוד
כאשר w=[0,0],b=0,x=[1,0],y=0 וקצב הלמידה 0.5, החיזוי הוא 1 והטעות −1. לאחר עדכון w=[−0.5,0] ו־b=−0.5.

</div>

In [ ]:
import numpy as np
w = np.array([0., 0.])
b = 0.
x = np.array([1., 0.])
y = 0
z = np.dot(w, x) + b
predicted = 1 if z >= 0 else 0
error = y - predicted
w = w + 0.5 * error * x
b = b + 0.5 * error
print(w, b)

<div dir="rtl" lang="he" style="text-align: right;">

**פלט צפוי:** [-0.5  0. ] -0.5

## מדוע זה חשוב?
סכום משוקלל, שגיאה ועדכון משקולות מכינים ל־DL. רגרסיה ופרספטרון אינם מוצגים כאלגוריתמי חובה ביחידה 5.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## יישום מודרך — חלק 1
מקור הקוד: ערכה 33 הקיימת. בהמשך עוברים לתרגול עצמאי באותו נושא.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# טבלת אמת מומצאת: AND
X=np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=np.array([0,0,0,1])

fixed_w=np.array([1.,1.]);fixed_b=-1.5
print("z for (1,1):",np.dot(fixed_w,[1,1])+fixed_b)
print("Boundary z=0 predicts:",1 if 0>0 else 0)

<div dir="rtl" lang="he" style="text-align: right;">

### קריאת פונקציה
זהו את הפרמטרים, את החישוב ואת הערך המוחזר. עקבו אחר קלט קטן לפני ההרצה. התבנית נשענת על המימוש שכבר נבדק; אין צורך במחלקה חדשה.

</div>

In [ ]:
def predict_perceptron(X,w,b):
    output=[]
    for row in X:
        z=float(np.dot(w,row)+b)
        output.append(1 if z>=0 else 0)
    return np.array(output)

def train_perceptron(X,y,learning_rate=0.5,max_epochs=40):
    w=np.zeros(X.shape[1],dtype=float)
    b=0.0
    trace=[]
    for epoch in range(max_epochs):
        mistakes=0
        for i in range(len(X)):
            before=w.copy()
            b_before=b
            z=float(np.dot(w,X[i])+b)
            predicted=1 if z>=0 else 0
            error=int(y[i])-predicted
            if error!=0:
                w=w+learning_rate*error*X[i]
                b=b+learning_rate*error
                mistakes=mistakes+1
            trace.append([epoch,i,z,predicted,int(y[i]),error,
                          before[0],before[1],b_before,w[0],w[1],b])
        if mistakes==0:
            break
    columns=["epoch","row","z","pred","actual","error","w1_before","w2_before","b_before","w1_after","w2_after","b_after"]
    return w,b,pd.DataFrame(trace,columns=columns)

w,b,trace=train_perceptron(X,y)
predictions=predict_perceptron(X,w,b)
display(trace.head(12))
print("Weights:",w,"Bias:",b,"Training predictions:",predictions)

<div dir="rtl" lang="he" style="text-align: right;">

### המחשה חזותית
לפני הרצה הגדירו שאלה, סוגי משתנים ויחידות. אחרי הרצה פרשו את הצירים ואת הממצא. אין להסיק סיבתיות או למחוק קצה רק מן הציור.

</div>

In [ ]:
plt.figure()
plt.scatter(X[:,0],X[:,1],c=y,cmap="coolwarm",vmin=0,vmax=1)
grid=np.array([-0.2,1.2])
if abs(w[1])>1e-12:
    plt.plot(grid,-(w[0]*grid+b)/w[1],color="black")
elif abs(w[0])>1e-12:
    plt.axvline(-b/w[0],color="black")
plt.xlim(-0.2,1.2);plt.ylim(-0.2,1.2)
plt.xlabel("x1");plt.ylabel("x2");plt.title("Synthetic training data — perceptron boundary")
plt.tight_layout();plt.show()
xor_y=np.array([0,1,1,0])
print("XOR labels:",xor_y)
# לא דורשים מהאלגוריתם למצוא קו שאינו קיים; ממשיכים לרשת בשיעור 35

<div dir="rtl" lang="he" style="text-align: right;">

## עצירה לפני המעבדה
מה התחזית ב־z=0 לפי הכלל כאן?

**התוצר הנדרש:** עוקבים אחר החלטה ועדכון אחד, ואז מסבירים תבנית אימון ותרגול OR. בכל החומרים החדשים משתמשים ב־>=0 באופן עקבי.

</div>

## בקרה מתמטית: סף ועדכון
לפי המקור הסף הוא z>=0. כאשר w=[0,0], b=0, x=[1,0] והתווית 0, החיזוי 1, השגיאה −1, והתיקון בקצב 0.5 הוא w=[−0.5,0], b=−0.5.

In [ ]:
assert np.array_equal(X,np.array([[0,0],[0,1],[1,0],[1,1]]))
assert y.tolist()==[0,0,0,1]
assert np.array_equal(predictions,y)
assert len(trace)>0
assert {"error","w1_before","w1_after","b_before","b_after"}.issubset(trace.columns)
assert (trace["actual"]-trace["pred"]==trace["error"]).all()
assert len(trace)<=40*4
assert np.isclose(.5*(-1),-.5)
assert (predict_perceptron(X,np.array([0.,0.]),0)==1).all()
print("AND manual/update and training verified")

## בקרת מסווג XOR
XOR אינו ניתן להפרדה ליניארית במרחב שני הקלטים. אין להציג את ההרצה כאילו מודל יחיד יכול ללמוד ארבע תחזיות XOR.

In [ ]:
assert xor_y.tolist()==[0,1,1,0]
assert np.array_equal(y,np.array([0,0,0,1]))
assert np.all(np.isin(predictions,[0,1]))
assert np.isfinite(w).all() and np.isfinite(b)
print("AND accuracy:",np.mean(predictions==y))